# 📦 Import Libraries



In [ ]:
# ── Standard Libraries ──────────────────────────────
import numpy as np
import pandas as pd
import re
from collections import Counter

# ── Visualization ────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Arial'
import seaborn as sns
from wordcloud import WordCloud

# ── Arabic NLP ───────────────────────────────────────
import arabic_reshaper
from bidi.algorithm import get_display
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

# ── Scikit-learn ─────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight

# ── TensorFlow / Keras ───────────────────────────────
import tensorflow as tf
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (
    Input, Embedding, SpatialDropout1D,
    Bidirectional, LSTM, BatchNormalization,
    Dense, Dropout
)

# 📂 Load Data


In [ ]:
df = pd.read_csv("../data/dataset.csv")

print(f"إجمالي عدد التغريدات: {len(df)}")
df.head()

# 🧹 Data Cleaning & Preprocessing

In [ ]:
df = df[['review_description', 'rating']].dropna()
df = df[df['rating'].isin([1, 0, -1])]
df['label'] = df['rating'].map({1: 'pos', 0: 'neu', -1: 'neg'})

In [ ]:
# Oversample neutral class only
neu_df = df[df['label'] == 'neu']
extra_neu = pd.concat([neu_df] * 9)  # أضف 9 نسخ إضافية = x10 للـ neu فقط

df = pd.concat([df, extra_neu]).reset_index(drop=True)

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(df['label'].value_counts())
print(f"Total reviews: {len(df)}")

## 🔍 Check & Remove Duplicates / Nulls


In [ ]:
# Check nulls and duplicates
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())

# Clean
df = df[['review_description', 'label']].dropna()
df = df.drop_duplicates()
df = df.reset_index(drop=True)
print(f"Total rows after cleaning: {len(df)}")

## ⚖️ Balance Classes (Oversample Neutral)

In [ ]:
# Oversample neutral class only
neu_df = df[df['label'] == 'neu']
extra_neu = pd.concat([neu_df] * 9)
df = pd.concat([df, extra_neu]).reset_index(drop=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(df['label'].value_counts())
print(f"Total reviews: {len(df)}")

In [ ]:
df.info()
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")
print(f"Columns names: {df.columns.tolist()}")
print(f"Data types:\n{df.dtypes}")
print(df['label'].value_counts())

## 📏 Text Length Analysis


In [ ]:
df['text_length'] = df['review_description'].apply(len)

# Stats
print(f"Max length : {df['text_length'].max()} chars")
print(f"Min length : {df['text_length'].min()} chars")
print(f"Avg length : {df['text_length'].mean():.1f} chars")

# Longest text
longest_text = df.loc[df['text_length'].idxmax(), 'review_description']
print(f"\nLongest review:\n{longest_text}")

# Top 5 longest
top5 = df.nlargest(5, 'text_length')[['review_description', 'text_length']].reset_index(drop=True)
for i, row in top5.iterrows():
    print(f"{'='*60}")
    print(f"Rank {i+1} | {row['text_length']} chars")
    print(f"{row['review_description']}")
print('='*60)

## 📊 Label Distribution Plot


In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(x='label', data=df, hue='label', palette='viridis', legend=False)

# إضافة الأرقام فوق كل عمود
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.title('Label Distribution (Positive vs Negative vs Neutral)')
plt.xlabel('Label')
plt.ylabel('Number of Reviews')
plt.tight_layout()
plt.show()

## 🧼 Arabic Text Cleaning


In [ ]:
def clean_arabic_text(text):
    text = re.sub(r"http\S+|www\S+|@\S+|#", "", text)       # Remove links/mentions
    text = re.sub(r"[\u064B-\u0652]", "", text)               # Remove tashkeel
    text = re.sub("[إأآا]", "ا", text)                        # Normalize alef
    text = re.sub("ى", "ي", text)                             # Normalize ya
    text = re.sub("ة", "ه", text)                             # Normalize ta marbuta
    text = re.sub("ؤ", "ء", text)                             # Normalize waw hamza
    text = re.sub("ئ", "ء", text)                             # Normalize ya hamza
    text = re.sub(r"[^\u0621-\u064A\s]", " ", text)           # Keep Arabic only
    text = re.sub(r"\s+", " ", text).strip()                  # Remove extra spaces
    return text

df['clean_text'] = df['review_description'].apply(clean_arabic_text)

## ☁️ Word Cloud


In [ ]:
text = ' '.join(df['clean_text'])
reshaped_text = arabic_reshaper.reshape(text)
bidi_text = get_display(reshaped_text)

wordcloud = WordCloud(
    width=1200, height=1200,
    background_color='black',
    font_path='arial.ttf',
    min_font_size=10
).generate(bidi_text)

plt.figure(figsize=(12, 12))
plt.imshow(wordcloud)
plt.axis("off")
plt.tight_layout(pad=0)
plt.show()

## 🚫 Remove Stopwords

In [ ]:
arabic_stopwords = set(stopwords.words('arabic'))

# Keep negation words — important for sentiment
negative_words = {'لا', 'ما', 'لم', 'ليس', 'لن', 'غير', 'بدون', 'لو', 'مو'}
arabic_stopwords -= negative_words

# Add extra stopwords
extra_words = {'تم', 'ان', 'كان', 'يكون', 'عن', 'بعد', 'قبل'}
arabic_stopwords.update(extra_words)

def remove_stopwords(text):
    words = text.split()
    return ' '.join([w for w in words if w not in arabic_stopwords])

df['clean_text'] = df['clean_text'].apply(remove_stopwords)
print(f"Total stopwords used: {len(arabic_stopwords)}")

## 🔢 Tokenization & Padding


In [ ]:
MAX_VOCAB = 60000
MAX_LEN   = 150

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_text'])

sequences = tokenizer.texts_to_sequences(df['clean_text'])
X_padded  = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

print(f"Matrix shape: {X_padded.shape}")

## 🏷️ Label Encoding


In [ ]:
le = LabelEncoder()
y  = le.fit_transform(df['label'])

print(f"Classes: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(df['label'].value_counts())

## ✂️ Train / Validation / Test Split

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_padded, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

## 🧠 Self-Attention Layer


In [ ]:
class SelfAttention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(SelfAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], 1),
                                 initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, x):
        e      = tf.tanh(tf.matmul(x, self.W) + self.b)
        a      = tf.nn.softmax(e, axis=1)
        output = tf.reduce_sum(x * a, axis=1)
        return output

## 🏗️ Model Architecture (BiLSTM + Attention)


In [ ]:
inputs = Input(shape=(MAX_LEN,))
x = Embedding(input_dim=MAX_VOCAB, output_dim=128)(inputs)
x = SpatialDropout1D(0.3)(x)
x = Bidirectional(LSTM(64, return_sequences=True, recurrent_dropout=0.4,
                       kernel_regularizer=regularizers.l2(0.01)))(x)
x = SelfAttention()(x)
x = BatchNormalization()(x)
x = Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.01))(x)
x = Dropout(0.4)(x)
outputs = Dense(3, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## ⚖️ Class Weights


In [ ]:
class_weights_arr = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
weights_dict = {
    0: 1.0,   # neg
    1: 6.0,   # neu — عاقبه أكثر على الأخطاء
    2: 1.0    # pos
}
print("Class weights:", weights_dict)

## 🏋️ Model Training


In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5,
                           restore_best_weights=True, verbose=1)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                               patience=2, min_lr=1e-6, verbose=1)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr],
    class_weight=weights_dict,
    shuffle=True,
    verbose=1
)

## 📈 Training History Plot


In [ ]:
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy During Training')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## 📊 Model Evaluation


In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['neg', 'neu', 'pos']))

## 🟦 Confusion Matrix


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted neg', 'Predicted neu', 'Predicted pos'],
            yticklabels=['Actual neg', 'Actual neu', 'Actual pos'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 🔮 Predict New Reviews


In [ ]:
def predict_sentiment(texts):
    cleaned = [clean_arabic_text(t) for t in texts]
    cleaned = [remove_stopwords(t) for t in cleaned]

    seq    = tokenizer.texts_to_sequences(cleaned)
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')

    predictions = model.predict(padded, verbose=0)
    labels = {0: 'Negative 😞', 1: 'Neutral 😐', 2: 'Positive 😊'}

    print("\n" + "="*50)
    for text, pred in zip(texts, predictions):
        class_idx = np.argmax(pred)
        label     = labels[class_idx]
        conf      = float(pred[class_idx])
        print(f"Text       : {text}")
        print(f"Sentiment  : {label}")
        print(f"Confidence : {conf*100:.1f}%")
        print("-"*50)

my_texts = [
    "هذا المنتج رائع جداً وسريع",
    "تجربة سيئة جداً ما أنصح فيه",
    "الخدمة ممتازة وسريعة الاستجابة",
    "بطيء ومزعج ما يستحق الشراء",
    "المنتج عادي لا بأس فيه",
]
predict_sentiment(my_texts)

## 💾 Save Model & Tokenizer


In [ ]:
import pickle, os

os.makedirs('../model', exist_ok=True)

# حفظ الموديل
model.save('../model/sentiment_model.keras')

# حفظ الـ tokenizer
with open('../model/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# حفظ الـ label encoder
with open('../model/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("✅ تم الحفظ!")